# PyTorch (рекомендуется использовать Google Colab)

Ранее мы рассматривали построение и обучение нейронных сетей в Python с использованием TensorFlow Keras. Другим популярным фреймворком для задач глубокого обучения является PyTorch — это библиотека с открытым исходным кодом, разработанная Facebook (ныне Meta).

### Основные особенности PyTorch:

* PyTorch основан на тензорах и имеет сильные стороны в использовании ускорения GPU. Он известен своей гибкостью и динамическим вычислительным графом, что делает его популярным среди исследователей.

* Динамический вычислительный граф: Граф определяется во время выполнения, что делает отладку и разработку более гибкими, особенно для исследовательских задач с изменяющейся структурой модели.

* Сильная поддержка GPU: Эффективно использует GPU для ускорения вычислений.

* Обширная экосистема: Включает библиотеки, такие как TorchVision для компьютерного зрения и TorchText для обработки естественного языка.

### Когда следует использовать PyTorch вместо TensorFlow:

* Исследования и быстрое прототипирование: Динамический граф PyTorch делает его более гибким для экспериментов и быстрого изменения архитектуры моделей в процессе исследования.

* Задачи, требующие высокой гибкости: Если ваша задача включает динамические изменения в модели или требует сложной логики потока управления, PyTorch может быть удобнее.

* Определенные области исследований: В некоторых областях исследований (например, обработка естественного языка) PyTorch исторически был более популярен


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
!pip install torchtext==0.6.0

In [ ]:
import torchtext
from torchtext import data
from torchtext.datasets import IMDB

In [ ]:
# spacy -- вспомогательная библоитека для токенизации текста, скачаем токенайзер для английского языка
! pip install spacy
! python -m spacy download en_core_web_sm

In [ ]:
from torchtext.data import Field, LabelField

## Пример с моделью классификации на PyTorch

Теперь рассмотрим, как можно решить задачу классификации, используя библиотеку PyTorch.

### Генерация данных для классификации (PyTorch)

Для демонстрации классификации с PyTorch снова сгенерируем синтетический набор данных.

In [ ]:
# Generate a synthetic classification dataset
X_cls_pt, y_cls_pt = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, n_classes=2, random_state=42)

# Split the generated classification dataset into training and testing sets
X_cls_train_pt, X_cls_test_pt, y_cls_train_pt, y_cls_test_pt = train_test_split(X_cls_pt, y_cls_pt, test_size=0.2, random_state=42)

# Scale the data (important for many models, including neural networks)
scaler_pt = StandardScaler()
X_cls_train_pt = scaler_pt.fit_transform(X_cls_train_pt)
X_cls_test_pt = scaler_pt.transform(X_cls_test_pt)

# Convert numpy arrays to PyTorch tensors
X_cls_train_pt = torch.tensor(X_cls_train_pt, dtype=torch.float32)
y_cls_train_pt = torch.tensor(y_cls_train_pt, dtype=torch.float32).unsqueeze(1) # Add an extra dimension for binary cross entropy
X_cls_test_pt = torch.tensor(X_cls_test_pt, dtype=torch.float32)
y_cls_test_pt = torch.tensor(y_cls_test_pt, dtype=torch.float32).unsqueeze(1) # Add an extra dimension

print("Форма X_cls_train_pt:", X_cls_train_pt.shape)
print("Форма X_cls_test_pt:", X_cls_test_pt.shape)
print("Форма y_cls_train_pt:", y_cls_train_pt.shape)
print("Форма y_cls_test_pt:", y_cls_test_pt.shape)

torch.tensor — это базовый строительный блок библиотеки PyTorch.
Он работает примерно как numpy.ndarray, но с ключевым преимуществом:
может храниться на GPU и участвовать в автоматическом дифференцировании (autograd).

torch.tensor — это многомерный массив (тензор), который может:

* храниться на CPU или GPU

* участвовать в вычислении градиентов

* быть частью нейросетей и функций потерь

* эффективно обрабатываться векторами и матрицами

Тензоры в PyTorch — это ядро всей работы:
модели, параметры весов, батчи данных — всё представлено в виде тензоров.

### Архитектура модели классификации (PyTorch)

В PyTorch архитектура модели определяется путем создания класса, наследующего от `torch.nn.Module`. Слои определяются в конструкторе `__init__`, а прямой проход данных через модель - в методе `forward`.

In [ ]:
# Define the neural network architecture for classification using PyTorch
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        # Define the layers
        self.layer_1 = nn.Linear(input_dim, 128) # Input layer to first hidden layer
        self.relu = nn.ReLU()                    # ReLU activation
        self.layer_2 = nn.Linear(128, 64)        # First hidden layer to second hidden layer
        self.output_layer = nn.Linear(64, 1)     # Second hidden layer to output layer
        self.sigmoid = nn.Sigmoid()              # Sigmoid activation for binary output

    def forward(self, x):
        # Define the forward pass
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.output_layer(x)
        x = self.sigmoid(x) # Apply sigmoid to the output
        return x

# Instantiate the model
input_dim_pt = X_cls_train_pt.shape[1]
model_cls_pt = BinaryClassifier(input_dim_pt)

print("Архитектура модели PyTorch:")
print(model_cls_pt)

### Компиляция модели классификации (PyTorch)

В отличие от TensorFlow/Keras, в PyTorch "компиляция" модели не является отдельным явным шагом с методом `.compile()`. Вместо этого мы вручную определяем функцию потерь (loss function) и оптимизатор (optimizer) после определения архитектуры модели.

#### Функция потерь: `torch.nn.BCEWithLogitsLoss` или `torch.nn.BCELoss`

Для бинарной классификации в PyTorch часто используется `torch.nn.BCELoss` (Binary Cross Entropy) или `torch.nn.BCEWithLogitsLoss`. Последняя более численно стабильна и объединяет Sigmoid и BCELoss.

#### Оптимизатор: `torch.optim`

Пакет `torch.optim` предоставляет различные алгоритмы оптимизации, такие как `SGD`, `Adam`, `RMSprop` и другие. Мы выбираем оптимизатор и передаем ему параметры нашей модели (`model.parameters()`) и скорость обучения (`lr`).

In [ ]:
# Define the loss function and optimizer
criterion_cls_pt = nn.BCELoss() # Binary Cross Entropy Loss for binary classification
optimizer_cls_pt = optim.Adam(model_cls_pt.parameters(), lr=0.001) # Adam optimizer

### Обучение модели классификации (PyTorch)

Процесс обучения в PyTorch требует написания цикла обучения вручную. Внутри этого цикла мы выполняем следующие шаги для каждой эпохи и каждого батча:
1.  **Прямой проход (Forward Pass):** Пропускаем входные данные через модель для получения предсказаний.
2.  **Вычисление функции потерь (Calculate Loss):** Сравниваем предсказания модели с истинными метками с помощью выбранной функции потерь.
3.  **Обнуление градиентов (Zero Gradients):** Перед обратным проходом необходимо обнулить градиенты, накопленные с предыдущей итерации.
4.  **Обратный проход (Backward Pass):** Вычисляем градиенты функции потерь по отношению к параметрам модели.
5.  **Шаг оптимизатора (Optimizer Step):** Обновляем веса модели на основе вычисленных градиентов и выбранного алгоритма оптимизации.

Также важно отслеживать метрики (например, точность) и производительность на валидационном наборе данных.

In [ ]:
# Train the PyTorch classification model
epochs_pt = 100
batch_size_pt = 32

# Create DataLoader for batching (optional but recommended for larger datasets)
from torch.utils.data import DataLoader, TensorDataset

train_dataset_pt = TensorDataset(X_cls_train_pt, y_cls_train_pt)
train_loader_pt = DataLoader(train_dataset_pt, batch_size=batch_size_pt, shuffle=True)

# For simplicity, we'll train on the whole training set here, but DataLoader is better for larger datasets
# Manual batching loop (if not using DataLoader)

print("Начало обучения модели классификации PyTorch...")

for epoch in range(epochs_pt):
    model_cls_pt.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    # Iterate over batches (if using DataLoader) or the whole dataset
    # For simplicity with this small dataset, we'll use the whole dataset directly
    # If using DataLoader: for inputs, labels in train_loader_pt: ...
    inputs = X_cls_train_pt
    labels = y_cls_train_pt

    # Forward pass
    outputs = model_cls_pt(inputs)
    loss = criterion_cls_pt(outputs, labels)

    # Backward and optimize
    optimizer_cls_pt.zero_grad() # Zero the gradients
    loss.backward()             # Perform backward pass
    optimizer_cls_pt.step()     # Perform a single optimization step

    # Calculate training accuracy for monitoring (optional)
    predicted = (outputs > 0.5).float()
    total_predictions += labels.size(0)
    correct_predictions += (predicted == labels).sum().item()

    running_loss += loss.item()

    # Print progress
    if (epoch + 1) % 10 == 0:
        train_accuracy = 100 * correct_predictions / total_predictions
        print(f'Epoch [{epoch+1}/{epochs_pt}], Loss: {running_loss/1:.4f}, Accuracy: {train_accuracy:.2f}%')

print("Обучение модели классификации PyTorch завершено.")

### Оценка модели классификации (PyTorch)

Для оценки модели в PyTorch мы переводим модель в режим оценки (`model.eval()`) и используем `torch.no_grad()` для отключения вычисления градиентов, так как они не нужны для оценки и инференса.

In [ ]:
# Evaluate the trained PyTorch classification model on the test set
model_cls_pt.eval() # Set the model to evaluation mode
with torch.no_grad(): # Disable gradient calculation
    outputs_test = model_cls_pt(X_cls_test_pt)
    loss_test = criterion_cls_pt(outputs_test, y_cls_test_pt)

    # Calculate accuracy
    predicted_test = (outputs_test > 0.5).float()
    correct_test = (predicted_test == y_cls_test_pt).sum().item()
    total_test = y_cls_test_pt.size(0)
    accuracy_test = correct_test / total_test

print(f"\nТестовая функция потерь (Binary Crossentropy) PyTorch: {loss_test.item():.4f}")
print(f"Тестовая точность (Accuracy) PyTorch: {accuracy_test:.4f}")

### Инференс модели классификации (PyTorch)

Для построения прогнозов на новых данных в PyTorch также используется режим оценки (`model.eval()`) и отключение градиентов (`torch.no_grad()`). Выход модели для бинарной классификации — это вероятность принадлежности к положительному классу.

In [ ]:
# Make predictions with the trained PyTorch classification model
model_cls_pt.eval() # Set the model to evaluation mode
with torch.no_grad(): # Disable gradient calculation
    predictions_cls_pt = model_cls_pt(X_cls_test_pt)

print("\n### Предсказания Классификации PyTorch (первые 5 примеров):")
# Print the first 5 predictions and corresponding actual labels
for i in range(5):
    # The output is the predicted probability of the positive class
    predicted_prob_pt = predictions_cls_pt[i].item()
    # Convert probability to predicted class label (0 or 1)
    predicted_label_pt = 1 if predicted_prob_pt > 0.5 else 0
    print(f"Предсказание {i+1}: Вероятность положительного класса = {predicted_prob_pt:.4f} | Предсказанная метка: {predicted_label_pt} | Истинная метка: {y_cls_test_pt[i].item()}")

# PyTorch на примере классификации отзывов с использованием RNN (дополнительно)

Рассмотрим задачу классификации текстовых последовательностей на примере  датасета IMDB (датасет содержит отзывы на фильмы, и необходимо определить их тональность - позитивные или негативные) с использованием библиотеки `torchtext`. В ней реализовано огромное число методов для обработки текстов.

__ВАЖНО__. Торчтекст не рекомендуется использовать для обучения на больших данных (от миллиона примеров и больше) из-за маленькой скорости работы. В таких случаях рекомендуется имплементировать свои датасеты.

## Модель

На вход модель будет принимать батч последовательностей токенов, а на выходе выдавать батч вероятностей классов.

Каждый токен пройдёт через эмбеддинг, а затем последовательность эмбеддингов пройдёт через LSTM. Самое последнее скрытое состояние будем считать векторным представлением для последовательности, поверх неё мы навесим линейный слой.

## Задание 1

Допишите конструктор класса. Инициализируйте слои:
* Embedding-слой, принимающий на вход num_embedding и выдающий векторы размера embedding_size

* LSTM-слой, принимающий на вход вектор размера embedding_size, имеющий скрытое состояние размера hidden_size. Также задайте batch_first=True

* Линейный слой, принимающий на вход вектор размера hidden_size и выдающий одно число.

In [ ]:
class TextClassifier(nn.Module):
    def __init__(
        self,
        num_embeddings=2502,
        embedding_size=300,
        hidden_size=200,
        num_classes=2,
        num_layers=1,
    ):
        super(TextClassifier, self).__init__()
        self.embedding = ...
        self.lstm = ...
        self.linear = ...

    def forward(self, x):
        embedded = self.embedding(x)
        _, (last_hidden, last_c) = self.lstm(embedded)
        return self.linear(last_hidden[0]).squeeze()


model = TextClassifier()

Запустите код ниже. Если он отработал без ошибок - все написано верно!

In [ ]:
model(torch.tensor([[1, 2, 3, 4], [1, 2, 3, 4]]))

## Скачаем и проинициализируем датасет

Дальше перейдём к торчтексту. Он имеет внутри себя коллекцию датасетов для разных задач NLP в том числе и отзывы на IMDB.



В `torchtext` существует такая сущность, как `Field`. Это просто класс, в котором содержится описание колонок нашего датасета. В нашем случае всё довольно просто. Есть две колонки - это текст и оценка, назовём их `text_field` и `label_field` соответственно. Токенизовать будем при помощи установленной только что библиотеки `spacy`, лейблы приведём к типу `float`.

In [ ]:
text_field = Field(
    tokenize="spacy",
    batch_first=True,
    include_lengths=False,
    tokenizer_language="en_core_web_sm",
)

label_field = LabelField(dtype=torch.float32, batch_first=True)

Разделим датасет на тренировочную и тестовую выборки при помощи метода splits.

In [ ]:
data_train, data_test = IMDB.splits(text_field, label_field)

## Задание 2

Посмотрите на первый (с индексом 0) отзыв в data_train. Какой он по тональности?

Подсказка: вам поможет `vars` — это встроенная функция Python, которая возвращает __dict__ атрибут объекта, содержащий его внутренние атрибуты. По сути, vars используется для получения всех свойств объекта в виде словаря.


In [ ]:
# ваше решение

Создадим также словари, соответствующие нашему тексту. Выкинем все слова (токены), которые встречаются редко. Оставим только 25000 самых частых слов.

Также согласуем номера токенов в словаре с эмбеддингами glove. При желании можно согласовать и с word2vec.

In [ ]:
vocab_size = 25000

# build_vocab -- создать словарь по данному полю в датасете
text_field.build_vocab(
    data_train,
    max_size=vocab_size,
    vectors="glove.6B.100d",
)

label_field.build_vocab(data_train)

for item in data_train:
    print(item.text)
    break

## Задание 3

Выведите 10 самых часто встречающихся токенов в словаре.
Какой токен третий по частоте встречаемости?

In [ ]:
# ваше решение

Сделаем аналог даталоадера.

In [ ]:
train_dataloader, test_dataloader = data.BucketIterator.splits(
    (data_train, data_test), batch_size=32, device="cuda:0"
)

## Задание 4

Возьмите любой батч из train_dataloader. Чему равна самая первая размерность тензора, описывающего поле `text` в батче?

In [ ]:
# ваше решение

Длина текста около 1000 символов это очень много для нашей маленькой модельки. Будем обрезать их на 256 токенах в нашем трейн лупе.

## Задание 5

#### Train loop

Построим train loop к нашей модели. Допишите цикл обучения.

In [ ]:
def train_epoch(
    model,
    data_loader,
    optimizer,
    criterion,
    return_losses=False,
    device="cuda:0",
):
    model = model.to(device).train()
    total_loss = 0
    num_batches = 0
    all_losses = []
    total_predictions = np.array([])  # .reshape((0, ))
    total_labels = np.array([])  # .reshape((0, ))

    with tqdm(total=len(data_loader), file=sys.stdout) as prbar:
        for item in data_loader:
            reviews = item.text
            labels = item.label

            # Move Batch to GPU
            reviews = reviews.to(device)
            labels = labels.to(device)
            predicted = ... # сделайте прогнозы при помощи модели
            loss = ... # вычислите loss

            # Update weights
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Update descirption for tqdm
            accuracy = ... # вычислите accuracy
            prbar.set_description(
                f"Loss: {round(loss.item(), 4)} "
                f"Accuracy: {round(accuracy.item() * 100, 4)}"
            )
            prbar.update(1)
            total_loss += loss.item()
            total_predictions = np.append(
                total_predictions, (predicted > 0.5).int().cpu().detach().numpy()
            )
            total_labels = np.append(total_labels, labels.cpu().detach().numpy())
            num_batches += 1
            all_losses.append(loss.detach().item())

    metrics = {"loss": total_loss / num_batches}
    metrics.update({"accuracy": (total_predictions == total_labels).mean()})
    if return_losses:
        return metrics, all_losses
    else:
        return metrics


def validate(model, data_loader, criterion, device="cuda:0"):

    model = model.eval()
    total_loss = 0
    num_batches = 0
    total_predictions = np.array([])
    total_labels = np.array([])

    with tqdm(total=len(data_loader), file=sys.stdout) as prbar:
        for reviews, labels in data_loader:
            reviews = reviews.to(device)
            labels = labels.to(device)
            predicted = model(reviews)

            loss = criterion(predicted, labels)
            accuracy = (predicted.argmax(1) == labels).float().mean()

            prbar.set_description(
                f"Loss: {round(loss.item(), 4)} "
                f"Accuracy: {round(accuracy.item() * 100, 4)}"
            )
            prbar.update(1)
            total_loss += loss.item()
            total_predictions = np.append(
                total_predictions, predicted.argmax(1).cpu().detach().numpy()
            )
            total_labels = np.append(total_labels, labels.cpu().detach().numpy())
            num_batches += 1

    metrics = {"loss": total_loss / num_batches}
    metrics.update({"accuracy": (total_predictions == total_labels).mean()})
    return metrics

## Задание 6

Попробуйте запустить обучение модели. Чему равен loss на третьей по счету эпохе?

Оставьте в значении loss два знака после запятой, остальные отбросьте.

In [ ]:
import sys

import numpy as np
from tqdm.notebook import tqdm

device = "cuda:0"
model = TextClassifier()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

for i in range(10):
    train_epoch(
        model, train_dataloader, criterion=criterion, optimizer=optimizer, device=device
    )

Кажется, обучается так себе. Давайте это исправлять!

Попробуем сделать следующее:

* сделаем модель двунаправленной

* добавим DropOut

* инициализируем эмбеддинги GloVe векторами

## Задание 7

Заполните метод `forward` класса `ModernTextClassifier`.

Запустите обучение модели.

In [ ]:
class ModernTextClassifier(nn.Module):
    def __init__(
        self,
        num_embeddings=25002,
        embedding_size=300,
        hidden_size=200,
        num_classes=2,
        num_layers=1,
        pad_token=1,
    ):
        super(ModernTextClassifier, self).__init__()
        self.embedding = nn.Embedding(
            num_embeddings, embedding_size, padding_idx=pad_token
        )
        self.lstm = nn.LSTM(
            embedding_size,
            hidden_size,
            batch_first=True,
            num_layers=num_layers,
            bidirectional=True,
        )
        self.linear = nn.Linear(hidden_size * num_layers, 1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # ваш код здесь


device = "cuda:0"
pad_token = text_field.vocab.stoi["<pad>"]
model = ModernTextClassifier(
    hidden_size=512, embedding_size=100, num_layers=2, pad_token=pad_token
)
model.embedding.weight.data = text_field.vocab.vectors

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.BCEWithLogitsLoss()

for i in range(10):
    train_epoch(
        model, train_dataloader, criterion=criterion, optimizer=optimizer, device=device
    )

model(torch.tensor([[1, 2, 3, 4], [1, 2, 3, 4]]))